# QuantLLMBot Phase 4 — Qwen2.5-14B QLoRA (Colab)

Runs the repo pipeline: preprocess → finetune → evaluate.

**Runtime:** GPU — **A100 40GB** recommended (L4 24GB works; T4 16GB: set fp16 and `max_seq_length` 2048 in `config.py`).

**Upload:** `QuantLLMBot_training.zip` (made on the local PC — contains `scripts/` + `model_training/knowledge/`).

Data: 4 stage JSONLs (115 train / 10 test). Output: LoRA adapter in `model_training/outputs/lora_weights/`.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload QuantLLMBot_training.zip and extract
from google.colab import files
up = files.upload()  # pick QuantLLMBot_training.zip from your PC

!unzip -o -q /content/QuantLLMBot_training.zip -d /content/QuantLLMBot

import os
os.environ['QUANTLLM_ROOT'] = '/content/QuantLLMBot'
%cd /content/QuantLLMBot/scripts

In [ ]:
# 3. Install dependencies (~2 min)
!pip install -q -r requirements.txt

In [ ]:
# 4. Pre-flight checks — everything must be green before cell 6
!python pre_training_checklist.py

In [ ]:
# 5. Preprocess — expect "115 instruction-response pairs"
!python 01_preprocess.py

In [ ]:
# 6. Fine-tune Qwen2.5-14B with QLoRA
# First run downloads ~15 GB of model weights, then trains (~30-60 min on A100)
!python 02_finetune.py

In [ ]:
# 7. Evaluate on the 10 held-out examples
!python 03_evaluate.py
!cat /content/QuantLLMBot/model_training/outputs/evaluation_results.json

In [ ]:
# 8. Download the trained adapter
!cd /content/QuantLLMBot/model_training/outputs && zip -qr qwen14b_lora_weights.zip lora_weights
from google.colab import files
files.download('/content/QuantLLMBot/model_training/outputs/qwen14b_lora_weights.zip')